In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
def optimize_fico_buckets(df, num_buckets=10):
    # 1. Prepare data: count defaults (k) and total records (n) per FICO score
    stats = df.groupby('fico_score')['default'].agg(['count', 'sum']).reset_index()
    stats.columns = ['fico', 'n', 'k']
    stats = stats.sort_values('fico')
    
    fico_values = stats['fico'].values
    n_vals = stats['n'].values
    k_vals = stats['k'].values
    m = len(fico_values)
    
    # 2. Log-Likelihood Helper Function
    def get_ll(start, end):
        total_n = np.sum(n_vals[start : end + 1])
        total_k = np.sum(k_vals[start : end + 1])
        if total_n == 0: return 0
        p = total_k / total_n
        ll = 0
        if p > 0: ll += total_k * np.log(p)
        if p < 1: ll += (total_n - total_k) * np.log(1 - p)
        return ll

    # 3. Dynamic Programming Table
    dp = np.full((num_buckets + 1, m), -np.inf)
    parent = np.zeros((num_buckets + 1, m), dtype=int)
    
    # Base case: 1 bucket
    for j in range(m):
        dp[1][j] = get_ll(0, j)
        
    # DP Transitions: Fill table for b buckets
    for b in range(2, num_buckets + 1):
        for j in range(b - 1, m):
            for i in range(b - 2, j):
                val = dp[b-1][i] + get_ll(i+1, j)
                if val > dp[b][j]:
                    dp[b][j] = val
                    parent[b][j] = i
                    
    # 4. Backtrack to find optimal boundaries
    boundaries = []
    curr_j = m - 1
    for b in range(num_buckets, 1, -1):
        split_idx = parent[b][curr_j]
        boundaries.append(fico_values[split_idx])
        curr_j = split_idx
        
    return sorted(boundaries)

# --- EXECUTION BLOCK ---
file_name = 'Loan_Data.csv'

if not os.path.exists(file_name):
    print(f"Error: {file_name} not found.")
else:
    loan_df = pd.read_csv(file_name)
    
    # Optimize for 10 buckets as requested by Charlie
    optimal_splits = optimize_fico_buckets(loan_df, num_buckets=10)
    
    print("--- Optimization Complete ---")
    print(f"Optimal FICO Boundaries: {optimal_splits}")
    
    # Example of how to apply the ratings
    # Rating 1 = Best (Highest FICO), Rating 10 = Worst (Lowest FICO)
    def assign_rating(fico):
        for i, b in enumerate(optimal_splits):
            if fico <= b:
                return 10 - i
        return 1

    loan_df['rating'] = loan_df['fico_score'].apply(assign_rating)
    print("\nSample of Assigned Ratings:")
    print(loan_df[['fico_score', 'rating']].head())

--- Optimization Complete ---
Optimal FICO Boundaries: [np.int64(520), np.int64(552), np.int64(580), np.int64(611), np.int64(649), np.int64(696), np.int64(732), np.int64(752), np.int64(753)]

Sample of Assigned Ratings:
   fico_score  rating
0         605       7
1         572       8
2         602       7
3         612       6
4         631       6
